# Exercise for Graph III: Sequential Graph Pipeline

### Your Task:
1. Accept a user's `name`, `age`, and a list of their `skills`.
2. Pass the state sequentially through three nodes:
   - **First node (`greet_user`)**: Personalizes the name field with a greeting.
   - **Second node (`describe_age`)**: Describes the user's age.
   - **Third node (`list_skills`)**: Lists the user's skills in a formatted string.
3. The final output in the `result` field should be a combined message formatted as:
   > `"Linda, welcome to the system! You are 31 years old! You have skills in: Python, Machine Learning, and LangGraph"`

**Hint:** You will need to use the `add_edge` method twice between the three processing nodes (`add_edge(node1, node2)` and `add_edge(node2, node3)`).

In [1]:
# Install dependencies if needed:
# !pip install langgraph langchain-core

## 1. Define State and Sequential Nodes

In [2]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

# 1. Define State
class UserProfileState(TypedDict):
    name: str
    age: int
    skills: List[str]
    result: str

# Node 1: Personalizes the name field with a greeting
def greet_user(state: UserProfileState):
    return {"result": f"{state['name']}, welcome to the system! "}

# Node 2: Describes the user's age
def describe_age(state: UserProfileState):
    return {"result": state["result"] + f"You are {state['age']} years old! "}

# Node 3: Lists the user's skills in a formatted string
def list_skills(state: UserProfileState):
    skills = state["skills"]
    if len(skills) > 1:
        formatted_skills = ", ".join(skills[:-1]) + ", and " + skills[-1]
    elif len(skills) == 1:
        formatted_skills = skills[0]
    else:
        formatted_skills = "None"
    return {"result": state["result"] + f"You have skills in: {formatted_skills}"}

## 2. Build the Sequential Graph

Connecting the three nodes sequentially:
- `START -> greet_user`
- `greet_user -> describe_age` *(First node-to-node edge)*
- `describe_age -> list_skills` *(Second node-to-node edge)*
- `list_skills -> END`

In [3]:
workflow = StateGraph(UserProfileState)

# Add the three nodes
workflow.add_node("greet_user", greet_user)
workflow.add_node("describe_age", describe_age)
workflow.add_node("list_skills", list_skills)

# Connect START to the first node
workflow.add_edge(START, "greet_user")

# Connect the sequential pipeline (using add_edge twice between intermediate nodes)
workflow.add_edge("greet_user", "describe_age")
workflow.add_edge("describe_age", "list_skills")

# Connect the final node to END
workflow.add_edge("list_skills", END)

# Compile the graph
app = workflow.compile()

## 3. Run and Verify with the Exercise Input

In [4]:
input_data = {
    "name": "Linda",
    "age": 31,
    "skills": ["Python", "Machine Learning", "LangGraph"]
}

output_state = app.invoke(input_data)

print("Final Result Message:")
print(output_state["result"])
print("\nFull State:")
print(output_state)

Final Result Message:
Linda, welcome to the system! You are 31 years old! You have skills in: Python, Machine Learning, and LangGraph

Full State:
{'name': 'Linda', 'age': 31, 'skills': ['Python', 'Machine Learning', 'LangGraph'], 'result': 'Linda, welcome to the system! You are 31 years old! You have skills in: Python, Machine Learning, and LangGraph'}
